# 第九阶段第一课：注意力机制

Transformer 的核心是注意力（Attention）。理解它只需要一个直觉：句子里的每个词，在看别的词时，应该"注意"到什么程度。这一课从零实现缩放点积注意力。

## 1. Q、K、V 是什么

每个词都会生成三个向量：
- Query（Q）：我想找什么
- Key（K）：我是什么
- Value（V）：我实际提供的内容

注意力分数 = Q 和 K 的相似度。相似度高的词，它的 V 会占更大权重。

In [ ]:
import torch

# 3 个词，每个词用 4 维向量表示
# 这里的 Q K V 是简化版，直接随机生成演示
Q = torch.randn(3, 4)   # 3 个词的查询
K = torch.randn(3, 4)   # 3 个词的键
V = torch.randn(3, 4)   # 3 个词的值
print(Q)

## 2. 计算注意力分数

分数 = Q 和 K 的点积，再除以 sqrt(d)（缩放，防止分数过大）。

In [ ]:
d = Q.shape[-1]
scores = Q @ K.T / (d ** 0.5)   # 3x3 的相似度矩阵
print(scores)

## 3. 转成权重：softmax

softmax 把分数变成概率分布（每行加起来等于 1），表示每个词注意谁。

In [ ]:
import torch.nn.functional as F

weights = F.softmax(scores, dim=-1)
print(weights)
print(weights.sum(dim=-1))     # 每行和为 1

## 4. 加权求和得到输出

用权重对 V 加权求和，得到每个词的注意力输出。

In [ ]:
output = weights @ V     # 3x4
print(output)

## 5. 封装成一个函数：缩放点积注意力

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d = Q.shape[-1]
    scores = Q @ K.T / (d ** 0.5)
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

out, w = scaled_dot_product_attention(Q, K, V)
print("输出形状：", out.shape)
print("注意力权重：")
print(w)

## 6. 多头注意力：多个"视角"

多头注意力 = 把 Q K V 切成几份，各算各的注意力，再拼起来。每个头关注不同类型的关系。

In [ ]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=16, n_head=4):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        batch, seq, _ = x.shape
        q = self.Wq(x).view(batch, seq, self.n_head, self.d_head).transpose(1, 2)
        k = self.Wk(x).view(batch, seq, self.n_head, self.d_head).transpose(1, 2)
        v = self.Wv(x).view(batch, seq, self.n_head, self.d_head).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_head ** 0.5)
        weights = F.softmax(scores, dim=-1)
        attn = weights @ v
        attn = attn.transpose(1, 2).contiguous().view(batch, seq, -1)
        return self.Wo(attn)

mha = MultiHeadAttention()
x = torch.randn(2, 5, 16)    # 2 句话，每句 5 个词，16 维
print(mha(x).shape)          # 形状不变：2, 5, 16

## 7. 练习（自己动手写）

练习 1：把上面的注意力函数抄一遍，输入 4 个词（Q K V 都是 4x8），确认输出是 4x8。

练习 2：解释一下为什么注意力要除以 sqrt(d)（提示：d 大时点积会很大，softmax 梯度会消失）。

In [ ]:
# 在这里写你的练习代码
